# ML-KEM (CRYSTALS-Kyber) — Postkvantový mechanismus zapouzdření klíče

**Předmět:** Kryptologie  |  **Implementace:** Python (od nuly, pouze se standardní knihovnou)

---


## 1. Historie a motivace

**ML-KEM** je standardizovaný název od organizace NIST pro algoritmus **CRYSTALS-Kyber**, který byl vydán jako standard **NIST FIPS 203** v srpnu 2024.

### Proč byl vytvořen?
Veškerá v současnosti nasazená asymetrická kryptografie (RSA, Diffie-Hellman, ECDH) spoléhá na výpočetní složitost rozkladu na prvočísla (faktorizace) nebo problému diskrétního logaritmu. Peter Shor v roce 1994 dokázal, že dostatečně velký kvantový počítač dokáže oba tyto problémy vyřešit v *polynomiálním* čase. Ačkoliv takový počítač dnes ještě neexistuje, hrozba útoků typu *„harvest-now, decrypt-later“* (zaznamenej nyní, dešifruj později – tedy plošné ukládání zašifrovaného provozu, dokud nebude kvantový počítač k dispozici) motivovala NIST ke standardizaci post-kvantových algoritmů dlouho předtím, než kvantový hardware dospěje.

### Časová osa
| Rok | Událost |
|------|-------|
| 2017 | Přihlášení algoritmu CRYSTALS-Kyber do PQC (Post-Quantum Cryptography) soutěže NIST |
| 2019 | Postup do 2. kola (z 26 kandidátů → 17) |
| 2020 | Postup mezi finalisty ve 3. kole |
| 2022 | Vybrán jako primární KEM algoritmus pro standardizaci |
| 2024 | Publikován jako oficiální standard **NIST FIPS 203** |

### Autoři
Roberto Avanzi, Joppe Bos, Léo Ducas, Eike Kiltz, Tancrède Lepoint, Vadim Lyubashevsky, John M. Schanck, Peter Schwabe, Gregor Seiler, Damien Stehlé.



## 2 Matematické základy

### Learning With Errors (LWE)
Máme-li dáno mnoho párů **(aᵢ, bᵢ)**, kde **bᵢ = ⟨aᵢ, s⟩ + eᵢ mod q**, je úkolem najít tajný vektor **s**.
Díky malému chybovému členu (šumu) **eᵢ** (vzorkovanému z centrované binomiální distribuce - CBD) se systém jeví jako k nerozeznání od náhody — věří se, že řešení tohoto problému je v nejhorším případě (worst-case) NP-těžké.

### Module-LWE (MLWE)
ML-KEM povyšuje LWE do polynomiálního kruhu **R_q = Z_q[X] / (X²⁵⁶ + 1)** a uspořádává polynomy do **k×k modulu** (pro ML-KEM-512 je k = 2). To umožňuje vytvořit mnohem kompaktnější klíče při zachování stejného argumentu o výpočetní složitosti (hardness).

### Number Theoretic Transform (NTT)
Násobení polynomů v R_q je úzkým hrdlem celého výpočtu. ML-KEM používá NTT, což je obdoba klasické rychlé Fourierovy transformace (FFT), ale nad tělesem Z_{3329}. Modul **q = 3329 = 13·2⁸ + 1** byl zvolen speciálně tak, aby existovala 256bodová NTT transformace. Díky NTT stojí násobení dvou polynomů stupně 255 pouze **O(n log n)** operací namísto původních O(n²).

### Schéma na nejvyšší úrovni (High-level scheme)
```text
KeyGen:   A ← náhodná matice v R_q^{k×k}
          s, e ← malé šumové polynomy
          t = A·s + e          (veřejný klíč = t, A;  tajný klíč = s)

Encaps:   zvol náhodné m ∈ {0,1}^256
          zašifruj m pomocí (t, A) → ciphertext c
          K = H(m)             (sdílené tajemství)

Decaps:   použij s k dešifrování m z c
          znovu odvoď K = H(m)
          FO kontrola re-enkrypce → implicitní zamítnutí (implicit rejection), pokud byl c zmanipulován

## 3 Pravidla použití

1. **Nikdy znovu nepoužívejte stejnou náhodnost** — `encaps` musí pro každé volání použít zcela nové náhodné 32bajtové semínko (seed).
2. **Udržujte `dk` v tajnosti** — kompromitování dekapsulačního (tajného) klíče prolomí všechny minulé i budoucí komunikační relace.
3. **Pravidelně klíče obměňujte (rotujte)** — doporučuje se provést maximálně 2³² zapouzdření (encapsulations) na jeden pár klíčů.
4. **Hybridní režim během přechodu** — NIST doporučuje párovat ML-KEM s klasickým asymetrickým algoritmem (tzv. KEM, např. X25519). Tím je zajištěno, že bezpečnost zůstane nedotčena i v případě, že by jedno ze schémat bylo nečekaně prolomeno.
5. **Používejte implicitní zamítnutí (implicit rejection) správně** — funkce `decaps` musí vždy vrátit *nějaký* klíč (nikdy nesmí vyhodit chybu). Pokud by hlásila chyby, stane se z ní dešifrovací orákulum, což útočníkovi otevře cestu k CCA útokům.
6. **Ověřujte velikost klíčů** — striktně odmítejte klíče s neočekávanou délkou, abyste zabránili útokům na snížení parametrů a úrovně bezpečnosti (parameter-downgrade attacks).

## 4 Důsledky porušení protokolu

| Porušení | Důsledek |
|-----------|-------------|
| Únik `dk` (tajného klíče) | Úplné prolomení: jakýkoliv ciphertext může být dekapsulován (dešifrován) |
| `decaps` vyhodí výjimku při špatném klíči | CCA2 orákulum → obnova tajného klíče |
| Slabý/opakovaně použitý RNG v `encaps` | Sdílené tajemství se stane předvídatelným |
| Vynechání vazby přes H(ek) (Domain separation) | Útok záměnou klíče (Key substitution attack): útočník podvrhne `ek` |
| Nepoužití KDF na sdílené tajemství | Strukturovaný výstup z G může prozradit částečné informace |
| Dlouhodobé používání bez rotace | Roste riziko útoků typu "harvest-now-decrypt-later" (zaznamenej nyní, dešifruj později) |

Nejkritičtějším porušením je **vystavení dešifrovacího orákula** (vracení chyby vs. OK).
Fujisaki-Okamoto (FO) transformace uvnitř ML-KEM tomuto specificky zabraňuje pomocí *implicitního zamítnutí* (implicit rejection): při manipulaci s ciphertextem funkce `decaps` tiše vrátí pseudonáhodnou hodnotu odvozenou ze skrytého semínka `z` — což je pro útočníka k nerozeznání od platného klíče.

## 5 Kryptoanalýza — Jak by mohl být ML-KEM prolomen?

### Redukce mřížek (Lattice reduction - nejlepší známý útok)
Veřejný klíč **t = As + e** definuje mřížku. Obnova vektoru **s** vyžaduje nalezení krátkého vektoru v této mřížce. Nejlepším praktickým algoritmem je **BKZ (Block Korkine-Zolotarev)**. Pro ML-KEM-512 vyžaduje potřebná velikost bloku algoritmu BKZ ≈ 2¹¹⁸ klasických operací — což je absolutně mimo dosah jakéhokoliv myslitelného počítače. Bezpečnost je dokázána v modelu QROM (Quantum Random Oracle Model).

### Kvantové útoky
- **Shorův algoritmus**: NELZE aplikovat — v problému MLWE není žádná struktura grupy, kterou by šlo zneužít.
- **Groverův algoritmus**: poskytuje √ (odmocninové) zrychlení při prohledávání; ML-KEM-512 cílí na 128bitovou klasickou / ~103bitovou kvantovou bezpečnost (NIST Kategorie 1).

### Útoky postranními kanály (Side-channel attacks)
Časové rozdíly při výpočtech "motýlkových operací" (butterfly operations) u NTT, vzorkování CBD nebo u FO kontroly re-enkrypce mohou prozradit tajné informace. Produkční implementace používají přísně **kód s konstantním časem (constant-time code)**. Tato výuková implementace operace v konstantním čase **nepoužívá**.

### Není znám žádný algoritmus v polynomiálním čase
Předpokládá se, že problém MLWE je v nejhorším případě (worst-case) NP-těžký (výpočetní složitost v průměrném případě /average-case/ matematicky vyplývá z redukce z nejhoršího na průměrný případ, kterou dokázal Lyubashevsky a spol.).

In [1]:
# Upload ml_kem.py to Colab session storage before running this cell.
# In Colab: Files (left panel) → Upload → select ml_kem.py
# Locally:  make sure ml_kem.py is in the same directory as this notebook.

import ml_kem
import hashlib

print('ML-KEM-512 parameters:')
print(f'  Polynomial degree  n = {ml_kem.N}')
print(f'  Modulus            q = {ml_kem.Q}')
print(f'  Module rank        k = {ml_kem.K}')
print(f'  Noise η1 / η2        = {ml_kem.ETA1} / {ml_kem.ETA2}')
print(f'  Compression du/dv    = {ml_kem.DU} / {ml_kem.DV}')
print()
print('Key / ciphertext sizes:')
print(f'  Encapsulation key (ek)  : {ml_kem.EK_SIZE} bytes')
print(f'  Decapsulation key (dk)  : {ml_kem.DK_PKE_SZ + ml_kem.EK_SIZE + 64} bytes')
ct_size = 32*ml_kem.DU*ml_kem.K + 32*ml_kem.DV
print(f'  Ciphertext              : {ct_size} bytes')
print(f'  Shared secret           : 32 bytes')

ML-KEM-512 parameters:
  Polynomial degree  n = 256
  Modulus            q = 3329
  Module rank        k = 2
  Noise η1 / η2        = 3 / 2
  Compression du/dv    = 10 / 4

Key / ciphertext sizes:
  Encapsulation key (ek)  : 800 bytes
  Decapsulation key (dk)  : 1632 bytes
  Ciphertext              : 768 bytes
  Shared secret           : 32 bytes


## 6  Demo — Key Generation


In [2]:
print('Generating ML-KEM-512 key pair ... (may take a few seconds)')
ek, dk = ml_kem.keygen()
print(f'Done.')
print(f'  ek ({len(ek)} bytes): {ek[:16].hex()}...')
print(f'  dk ({len(dk)} bytes): {dk[:16].hex()}...')

Generating ML-KEM-512 key pair ... (may take a few seconds)
Done.
  ek (800 bytes): 9131b983d189e105b3ad45ce5873683d...
  dk (1632 bytes): 34e28e8cc090ecc5a9a4987e93500d58...


## 7  Demo — Encrypt Czech Text


In [3]:
plaintext = (
    'Kybernetická kryptografie chrání digitální komunikaci před odposlechem a manipulací. '
    'ML-KEM je postkvantový algoritmus standardizovaný organizací NIST v roce 2024, jehož '
    'bezpečnost je postavena na matematickém problému Module Learning With Errors — tento '
    'problém neumí efektivně řešit ani kvantové počítače. Na rozdíl od klasických algoritmů '
    'jako RSA nebo ECDH, které by kvantový počítač prolomil za polynomiální čas, ML-KEM '
    'odolává i budoucím kvantovým útokům. Proto se postupně zavádí do síťových protokolů '
    'jako TLS, SSH a VPN.'
)

print('Plaintext:')
print(plaintext)
print()

c_kem, ciphertext = ml_kem.encrypt_text(ek, plaintext)

print(f'KEM ciphertext ({len(c_kem)} bytes): {c_kem[:16].hex()}...')
print(f'Encrypted text ({len(ciphertext)} bytes, hex):')
print(ciphertext.hex())

Plaintext:
Kybernetická kryptografie chrání digitální komunikaci před odposlechem a manipulací. ML-KEM je postkvantový algoritmus standardizovaný organizací NIST v roce 2024, jehož bezpečnost je postavena na matematickém problému Module Learning With Errors — tento problém neumí efektivně řešit ani kvantové počítače. Na rozdíl od klasických algoritmů jako RSA nebo ECDH, které by kvantový počítač prolomil za polynomiální čas, ML-KEM odolává i budoucím kvantovým útokům. Proto se postupně zavádí do síťových protokolů jako TLS, SSH a VPN.

KEM ciphertext (768 bytes): 8335731b5f152fad3fc889a747084992...
Encrypted text (578 bytes, hex):
ef7ac6007a5e13eb8b48aa852674a357b82c49dda6d345f6fd912483e9d3783de4e12447870b358329e3079d3f6a66dc5ae9d40ad153b5067207527198dc1a56d30f6ea201d6d5c2cb7d2ddb07c04e88598e6acb6aa7cc8299a6cdc5a811b48a9f81829c584c68b2d35ccc1e310f17a4c935ca43c09e2cc1ece21802d6fd64dcb90babde225645f48bd86dc8c0cf2612dc6393aff6a55cab3b53cef1f3f5e0d2d3456407030d63b599de6a57acbd81ecc25f40d2a

## 8  Demo — Decryption with Correct Key


In [4]:
decrypted = ml_kem.decrypt_text(dk, c_kem, ciphertext)
print('Decrypted text:')
print(decrypted)
print()
print('Match:', decrypted == plaintext)

Decrypted text:
Kybernetická kryptografie chrání digitální komunikaci před odposlechem a manipulací. ML-KEM je postkvantový algoritmus standardizovaný organizací NIST v roce 2024, jehož bezpečnost je postavena na matematickém problému Module Learning With Errors — tento problém neumí efektivně řešit ani kvantové počítače. Na rozdíl od klasických algoritmů jako RSA nebo ECDH, které by kvantový počítač prolomil za polynomiální čas, ML-KEM odolává i budoucím kvantovým útokům. Proto se postupně zavádí do síťových protokolů jako TLS, SSH a VPN.

Match: True


## 9  Demo — Decryption with Modified Key

We flip **one bit** in the secret key portion of `dk`.
ML-KEM's implicit rejection returns a *pseudorandom* value instead of signalling an error.
The resulting wrong shared secret produces pure garbage when used to decrypt.


In [5]:
# Flip bit 0 of the first byte of dk (inside the s_hat secret key part)
dk_bad = bytearray(dk)
dk_bad[0] ^= 1
dk_bad = bytes(dk_bad)

decrypted_bad = ml_kem.decrypt_text(dk_bad, c_kem, ciphertext)

print('Decryption with modified dk:')
print(decrypted_bad)
print()

# Verify the shared secrets are different
K_correct = ml_kem.decaps(dk,     c_kem)
K_wrong   = ml_kem.decaps(dk_bad, c_kem)
print(f'Shared secret (correct key) : {K_correct.hex()}')
print(f'Shared secret (wrong key)   : {K_wrong.hex()}')
print(f'Keys match: {K_correct == K_wrong}')

Decryption with modified dk:
[DECRYPTION FAILED — garbage bytes]: 058276f75e313567fcb428ae0250a3a3a2d69a1393a210fe08a150ca8464f450eefa4e07fac2aab7...

Shared secret (correct key) : 83408d12259c80f3b809335eb3a22948c2c92f0218182897206fb01414e2231a
Shared secret (wrong key)   : 7b4d6da7f3be863f14b825577f5cef9e903fc6b4c75d62c7e12c6baf23e9da60
Keys match: False


## 10  Demo — Second Text (Different Message, Same Key Pair)

The same key pair can be used multiple times — each `encaps` call generates fresh randomness,
so each ciphertext is independent.


In [6]:
plaintext2 = (
    'Postkvantová kryptografie je odpovědí na hrozbu, kterou představují kvantové počítače '
    'pro současné šifrovací systémy. Algoritmy jako ML-KEM, ML-DSA a SLH-DSA byly vybrány '
    'organizací NIST po šestileté soutěži a standardizovány v roce 2024. Jejich implementace '
    'do stávající infrastruktury probíhá postupně a vyžaduje hybridní přístup kombinující '
    'klasické a postkvantové algoritmy.'
)

c_kem2, ct2 = ml_kem.encrypt_text(ek, plaintext2)
decrypted2  = ml_kem.decrypt_text(dk, c_kem2, ct2)

print('Original : ', plaintext2[:60], '...')
print('Decrypted:', decrypted2[:60], '...')
print('Match:', decrypted2 == plaintext2)
print()
print('Note: ciphertexts are different even though same key pair was used.')
print(f'  c_kem  first 8 bytes: {c_kem[:8].hex()}')
print(f'  c_kem2 first 8 bytes: {c_kem2[:8].hex()}')

Original :  Postkvantová kryptografie je odpovědí na hrozbu, kterou před ...
Decrypted: Postkvantová kryptografie je odpovědí na hrozbu, kterou před ...
Match: True

Note: ciphertexts are different even though same key pair was used.
  c_kem  first 8 bytes: 8335731b5f152fad
  c_kem2 first 8 bytes: 5e18bc4ed774fa58
